# GOLD ATP TOURNAMENTS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("dim_tournaments").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
atp_tournaments = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
)

## Tournaments

In [9]:
from datetime import datetime

df = (
    atp_tournaments
    .join(atp_matches, "TOURNEY_ID", 'left')
    .withColumn(
        "TOURNEY_STATUS",
        f.when(
            f.max(
                f.when(f.col("MATCH_ROUND").isin("F", "RR"), 1)
                .when(f.col("REF_YEAR") < datetime.now().year, 1).otherwise(0)).over(Window.partitionBy("TOURNEY_ID")) == 1,
            "Finished"
        ).otherwise("In Progress")
    )
    .select(
        f.col("TOURNEY_ID"),
        f.col("TOURNEY_NAME"),
        f.when(f.col("TOURNEY_LEVEL") == "G", "Grand Slam")
            .when(f.col("TOURNEY_LEVEL") == 'M', 'Masters 1000')
            .when(f.col("TOURNEY_LEVEL") == '500', 'ATP 500')
            .when(f.col("TOURNEY_LEVEL") == '250', 'ATP 250')
            .when(f.col("TOURNEY_LEVEL") == 'A', 'Outro')
            .when(f.col("TOURNEY_LEVEL") == 'F', 'ATP Finals')
            .when(f.col("TOURNEY_LEVEL") == 'O', 'Olympics Games')
            .when(f.col("TOURNEY_LEVEL") == 'D', 'Davis Cup')
            .otherwise(f.lit("-")
        ).alias("TOURNEY_LEVEL"),
        f.coalesce(f.col("TOURNEY_DRAW_SIZE"), f.lit(-1)).cast('int').alias("TOURNEY_DRAW_SIZE"),
        f.coalesce(f.col("TOURNEY_SURFACE"), f.lit('-')).alias("TOURNEY_SURFACE"),
        f.when(f.col("TOURNEY_IS_INDOOR") == True, 'Sim')
            .when(f.col('TOURNEY_IS_INDOOR') == False, 'Não')
            .otherwise(f.lit("-")
        ).alias("TOURNEY_IS_INDOOR"),
        f.col("TOURNEY_STATUS"),
        f.coalesce(f.col("TOURNEY_START_DATE"), f.lit('-')).alias("TOURNEY_START_DATE"),
        f.coalesce(f.col("REF_YEAR"), f.lit('-')).alias("REF_YEAR")
    )
    .distinct()
)

## Save dataframe

### Local

In [10]:
df.toPandas().to_csv(
    r"../../../data/gold/dimension/dim_tournaments.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)